[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://github.com/codespaces)


<!-- gen:values.yml#weeks.05.notebook-title|Quest 05: The Pivot Pitch -->
# Week 5 Quest: The Pivot Pitch
<!-- /gen -->

*This week, your job is to describe the shape you **want** your data in.*

**This [quest](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-quest) takes about 90 minutes**. Work with your [Associate](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-associate) throughout. The [glossary](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md) explains each stage.

**Your transcript.** [SpecStory](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-specstory) saves every exchange with your Associate to `.specstory/history/`. Before you start, check that SpecStory is recording. The [workstation guide](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/codespaces-start.md) shows how.

## Stage 1: Briefing

*(about 5–10 minutes)*

> **From:** A. Partner, [Runnymede Analytics](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-runnymede-analytics)
> **To:** You ([Junior Analyst](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-analyst))
> **Re:** Marlow & Finch: One table for the board
>
> Marlow & Finch run twelve home-and-gifts stores across three regions: North, Midlands, South. For eighteen months they have run weekend promotions. The promotions are not cheap. At Friday's board meeting someone will ask the question their commercial director, **Rosa Finch**, put to me on the phone this morning:
>
> *"Are the weekend promotions actually working?"*
>
> She wants **one table** for the board pack. Not a dashboard, not a deck. A non-technical board must be able to read it in thirty seconds and trust it. We have their store-level daily sales with a promo flag. Build the table. Make sure it answers her question, not an easier one.
>
> One warning: Weekends are different from weekdays. Do not compare promotion days against ordinary Tuesdays and call the difference an effect. Do not let your Associate do it either.
>
> A.P.

**The business question:** "*Do weekend promotions lift sales, and is that true in every region?*"

**The deliverable:** A single table in preparation for the board meeting, plus *three* sentences you would use to comment on it.

**[Definition of Done](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-definition-of-done)**: Tick these yourself before claiming any [badge](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-badge):

- [ ] Every ✏️ prediction cell in [First look](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-first-look) filled in *before* its code cell was run
- [ ] A region × promo table of **average daily sales**, with one cell verified by an independent hand check
- [ ] A comparison that is **like-for-like** (weekends against weekends), chosen and defended over the naive one
- [ ] The till-feed bug in [Red Flags](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-red-flags) found, explained in business terms, and its size **quantified**
- [ ] Every [Debrief](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-debrief) prompt answered, and your SpecStory transcript confirmed in `.specstory/history/`

**Badges:**

- 🥉 A verified region × promo pivot table
- 🥈 A defended judgement between two analyses, refined into the board table
- 🥇 An extension of your own design (trend, store-level, or a better board table)

### The data

Run the cell below first. It generates Marlow & Finch's data and writes the following files:

- `mf_daily_sales.csv`: **One row per store per day**, with `date`, `store`, `town`, `region`, `promo` (True/False), and `sales` (£). This is your working file
- `mf_till_feed.csv`: The raw till export the daily file was built from. You need it only in Red Flags

The generator code contains the answer. Reading it is allowed, but it spoils the quest.

In [ ]:
# --- Data setup: Run this cell first (runs anywhere, no download needed) ---
import os
import numpy as np
import pandas as pd

DATA_DIR = "assets/data/week05"
os.makedirs(DATA_DIR, exist_ok=True)
rng = np.random.default_rng(5813)  # module seed: Do not change

# Marlow & Finch: 12 home-and-gifts stores in 3 regions, daily takings,
# 18 months (2025-01-01 to 2026-06-30), with a promotion flag per store-day.
REGIONS = {
    "North":    ["Harrogate", "Ilkley", "Skipton", "York"],
    "Midlands": ["Leamington", "Ludlow", "Stamford", "Buxton"],
    "South":    ["Marlow", "Henley", "Guildford", "Lewes"],
}
lift = {"North": 1.15, "Midlands": 1.00, "South": 1.12}

dates = pd.date_range("2025-01-01", "2026-06-30", freq="D")
frames, store_id = [], 100
for region, towns in REGIONS.items():
    for town in towns:
        store_id += 1
        base = rng.uniform(2600, 4200)
        weekend = dates.dayofweek >= 5
        promo = rng.random(len(dates)) < np.where(weekend, 0.45, 0.04)
        season = 1 + 0.15 * np.cos(2 * np.pi * (dates.dayofyear - 355) / 365)
        sales = (base * season
                 * np.where(weekend, 1.28, 1.0)
                 * np.where(promo, lift[region], 1.0)
                 * rng.normal(1, 0.06, len(dates)))
        frames.append(pd.DataFrame({"date": dates, "store": f"MF-{store_id}",
                                    "town": town, "region": region,
                                    "promo": promo, "sales": sales.round(2)}))

daily_data = pd.concat(frames, ignore_index=True)
daily_data.to_csv(f"{DATA_DIR}/mf_daily_sales.csv", index=False)

# The raw till export: On promo days the till writes TWO rows per store.
# Promo-priced lines and standard lines land on separate streams.
promo_rows = daily_data[daily_data["promo"]]
share = rng.uniform(0.35, 0.55, len(promo_rows))
promo_lines = promo_rows.assign(stream="promo_lines",
                                sales=(promo_rows["sales"] * share).round(2))
standard_lines = promo_rows.assign(stream="standard_lines",
                                   sales=(promo_rows["sales"] - promo_lines["sales"]).round(2))
till_feed = (pd.concat([daily_data[~daily_data["promo"]].assign(stream="standard_lines"),
                        promo_lines, standard_lines], ignore_index=True)
             .sample(frac=1, random_state=5813)
             .reset_index(drop=True))
till_feed.to_csv(f"{DATA_DIR}/mf_till_feed.csv", index=False)

print(f"Created {DATA_DIR}/mf_daily_sales.csv ({len(daily_data)} rows, one per store per day)")
print(f"Created {DATA_DIR}/mf_till_feed.csv ({len(till_feed)} rows, raw till streams)")

## Stage 2: First look

*(about 10–15 minutes)*

Your Associate drafted the cells below overnight. **[PREDICT-RUN](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-predict-run):** Before you run each cell, write one line in the ✏️ cell above it saying what it will output. Then run it and note any surprise. If a function is new to you, ask your Associate what it does *before* you write your prediction.

> ✏️ **Your prediction:** *(one line: What will this cell output? Write it BEFORE running.)*
>
> 

In [ ]:
# First look 1: What did the client actually send?
daily = pd.read_csv(f"{DATA_DIR}/mf_daily_sales.csv", parse_dates=["date"])
print(daily.shape)
print(daily.dtypes)
daily.head()

> ✏️ **Your prediction:** *(one line: What will this cell output? Write it BEFORE running.)*
>
> 

In [ ]:
# First look 2: When do promotions actually run?
# (.dt.dayofweek numbers the days Monday=0 … Sunday=6, so >= 5 means
#  Saturday or Sunday; np.where picks "weekend" where that is True)
daily["day_type"] = np.where(daily["date"].dt.dayofweek >= 5,
                             "weekend", "weekday")
# crosstab counts the rows in each day_type × promo combination
pd.crosstab(daily["day_type"], daily["promo"])

> ✏️ **Your prediction:** *(one line: What will this cell output? Write it BEFORE running.)*
>
> 

In [ ]:
# First look 3: Average sales by region and promo flag (note the SHAPE of the result)
daily.groupby(["region", "promo"])["sales"].mean().round(0)

> ✏️ **Your prediction:** *(one line: What will this cell output? Write it BEFORE running.)*
>
> 

In [ ]:
# First look 4: Is the promo programme even comparable across regions?
# (promo is True/False, and summing a boolean column counts the Trues,
#  so these cells are counts of promo store-days, not sales)
daily.pivot_table(values="promo", index="region", aggfunc="sum")

**First look findings.** Take these into Build:

1. First look 2 shows the Partner's warning in numbers. Promotions run almost entirely on **weekends**. So a comparison of promo days with no-promo days also compares weekends with weekdays, unless you design the comparison to avoid it
2. First look 3 returns a **`MultiIndex` `Series`**, a single column of numbers in which each row carries two labels, region and promo. It is not yet a board table. `unstack()` or `pivot_table()` turns it into a grid
3. First look 4 shows that the three regions run promotions at similar rates. If one region shows no effect, it is not because that region barely ran any

## Stage 3: Build

*(about 35–45 minutes)* <!-- values-guard: allow: a quest stage estimate, not the lecture length -->

Build has three tiers: [Bronze](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-bronze), [Silver](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-silver), and [Gold](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-gold). Your Associate cannot do the judging for you.

### 🥉 Bronze

**Deliverable:** A [pivot table](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot_table.html) of **average daily sales**, with regions as rows and promo/no-promo as columns, rounded to whole pounds. Hand check one cell.

This is a **[GENERATE-JUDGE](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-generate-judge)** job. The prompt is a *shape sentence* that names the `aggfunc`, the setting that tells `pivot_table()` how to combine the numbers in each cell. Here is a suggested prompt for your Associate:

> *"I have a `DataFrame` `daily` with columns date, store, town, region, promo (boolean), sales, day_type. Build a pivot table: Rows = region, columns = promo, cells = average daily sales, rounded to whole pounds. State the `aggfunc` explicitly."*

Then **check it before you hand it over**:

1. **Shape check:** The table has 3 rows and 2 columns. All six cells are filled, and none shows `NaN` (pandas' marker for a missing value)
2. **Hand check one cell:** Run `daily[(daily["region"] == "North") & daily["promo"]]["sales"].mean()` yourself. Round your answer to whole pounds and confirm it matches the table's North/True cell
3. **Sanity check:** Promo cells should be higher than no-promo cells in every region, and they will be. First look 2 shows why that is *not* yet evidence that promotions work

In [ ]:
# 🥉 Bronze: Build it here with your Associate, then verify it.
# TODO: Pivot table with rows = region, columns = promo, cells = average daily
#       sales (aggfunc stated explicitly), rounded to whole pounds.
# TODO: Hand-check the North/True cell with a filtered .mean().

pass

### 🥈 Silver

While you built [Bronze](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-bronze), your Associate drafted **two competing answers** to Rosa Finch's question. Run both. They disagree about the Midlands.

**Draft A** compares all promo days against all non-promo days:

In [ ]:
# The Associate's Draft A: "promotions work everywhere"
draft_a = daily.pivot_table(values="sales", index="region",
                            columns="promo", aggfunc="mean").round(0)
draft_a.columns = ["no_promo", "promo"]
draft_a["uplift_%"] = ((draft_a["promo"] / draft_a["no_promo"] - 1) * 100).round(1)
print(draft_a)
print("\nDraft A's conclusion: promotions lift sales in ALL regions."
      " Recommend extending the programme chain-wide.")

**Draft B** compares like with like, promo weekends against ordinary weekends, using a pivot with two column levels:

In [ ]:
# The Associate's Draft B: "compare weekends with weekends"
draft_b = daily.pivot_table(values="sales", index="region",
                            columns=["day_type", "promo"],
                            aggfunc="mean").round(0)
print(draft_b)

weekend = draft_b["weekend"]
weekend_uplift = ((weekend[True] / weekend[False] - 1) * 100).round(1)
print("\nWeekend-matched uplift (%) by region:")
print(weekend_uplift)

**Your [Silver](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-silver) task:** Judge between the two drafts. Do not pick one without a reason.

1. Explain *why* the two drafts disagree. First look 2 is the clue. What else, besides the promotion, differs between promo days and other days in Draft A? What does the weekend column of Draft B hold constant?
2. Paste both drafts into a chat with your Associate. Then ask *"argue for A, then argue for B, then say which one answers 'are weekend promotions working?' and why"*. Check its reasoning against the numbers
3. Choose one and **refine it into the board table**, a small grid a non-technical board can read in thirty seconds. Describe the shape you want to your Associate:
   - Give the columns plain names, not `True`/`False`
   - Round the sales to whole pounds
   - Add an uplift column
   - Add a **promo days** count per region, built with [**named aggregation**](https://pandas.pydata.org/docs/user_guide/groupby.html#named-aggregation) (`.agg(promo_days=("promo", "sum"))`). The board's first question is *"based on how many days?"*
   - Sort the regions so the pattern is easy to see
4. Defend your choice in the Markdown cell below the code. Write the **three sentences** you would say to the board while the table is on screen. The answer differs by region. Make sure your sentences say so

In [ ]:
# 🥈 Silver: Your board table, refined from the draft you chose.
# TODO: Readable column names, whole pounds, an uplift column, a caption-worthy
#       structure. Direct your Associate with a shape sentence; verify a cell.
# TODO: Add a 'promo days' count per region via NAMED AGGREGATION:
#       groupby("region").agg(promo_days=("promo", "sum")).

pass

**✍️ My choice and defence:** *(Say which draft you chose, and why it answers the director's question and the other draft does not. Then write the three sentences you would say to the board. Use your own words, not your Associate's transcript.)*

> 

### 🥇 Gold

Rosa Finch, on the phone again: *"The board will believe the table more if it moves. Or if it names names."* Pick **one** extension, or design your own:

- **Trend:** A 28-day rolling mean of daily sales per region, meaning each day averaged with the 27 days before it. Build it with `groupby`/`transform`, or a pivot then `rolling`. Plot it so the promo effect shows against the seasonal pattern. Does the uplift persist across the 18 months, or fade?
- **Names:** Weekend-matched uplift **per store** (12 rows, sorted). Does every store in a "working" region show the uplift, or do a few stores account for most of it? Which single store would you visit next week, and why?
- **A better board table:** Your own design, under a title that states the finding. For example, regions could be rows, with columns for weekend baseline, weekend promo, uplift %, and promo-day count. If you know hvPlot, an interactive version is welcome

State your design in the opening comment of the cell below. [Gold](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-gold) ends when you can say, in one sentence, what your extension adds to the board's <!-- plain-language: everyday decision -->*decision*, not only to the notebook.

In [ ]:
# 🥇 Gold. My extension: (state it here in a comment)
# What it adds to the board's decision:
# TODO: Design and build with your Associate.

pass

## Stage 4: Red Flags

*(about 10–15 minutes)*

**[BUG-HUNT](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-bug-hunt).** Late on Thursday, your Associate rebuilds the headline table from the raw till feed instead of the daily summary. Its reason is *"for accuracy"*. Its conclusion reverses everything you found. It runs without an error. Run it.

In [ ]:
# The Associate's till-feed version. It runs. Is it right?
till = pd.read_csv(f"{DATA_DIR}/mf_till_feed.csv", parse_dates=["date"])

board_v2 = till.pivot_table(values="sales", index="region",
                            columns="promo").round(0)
board_v2.columns = ["no_promo", "promo"]
board_v2["uplift_%"] = ((board_v2["promo"] / board_v2["no_promo"] - 1) * 100).round(1)
print(board_v2)
print("\nAssociate's conclusion: promotions REDUCE sales in every region."
      " Recommend cancelling the programme immediately.")

**Your audit:**

1. **Find it.** The comments in the data-setup cell explain how the till feed is laid out on promo days. Check `till["stream"].value_counts()`, or count rows per store-day. Then look at the pivot call: Which argument is *missing*, and what does `pivot_table()` do about it without telling you?
2. **Explain it in business terms** in the cell below. Do not stop at "the aggfunc is wrong". What does averaging over *rows* rather than *days* do when every promo day is split across two rows? What would the board have decided if they had seen this table?
3. **Fix it by directing, not typing.** The grain of a table is what one row of it stands for. Tell your Associate the till feed's grain, and the grain the board table must have. Have *it* write the correction (collapse to one row per store-day first, then pivot). Then **quantify the bug**: For each region, how many percentage points of "uplift" separate the wrong table from the right one? Confirm the fixed till-feed table matches your Bronze table

**✍️ The bug, in business terms:** *(Two or three sentences: What did the table actually compare, and what <!-- plain-language: everyday decision -->decision would the board have taken on it?)*

> 

In [ ]:
# 🚩 Fixed till-feed table: Corrected by directing your Associate.
# TODO: Collapse till to one row per store-day (groupby + sum), then pivot with
#       an explicit aggfunc. Quantify: Wrong uplift_% vs fixed uplift_%, by region.

pass

## Stage 5: Debrief

*(about 10–15 minutes)*

**[TEACH-BACK](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-teach-back).** Answer each prompt in the cells below. Then copy your answers into your `QuestLog.md` as this week's entry, dated, with the badge you claim.

**1. Client explanation:** "Explain this week's key technique to the client in three sentences, no jargon." *(For Quest 05: What is a pivot table, and why did comparing weekends with weekends change the answer?)*

> 

**2. Associate audit:** "Where was your Associate wrong, misleading, or unhelpful this week? Paste the exchange." *(The till-feed table counts, but an exchange from your own transcript in `.specstory/history/` is better. It is searchable.)*

> 

**3. Quiz me:** "Ask your Associate to set you three questions on this week's material. Paste the quiz and answer it *without* AI help." *(Suggested prompt: "Set me three short questions (easy, medium, hard) on `melt()`, `pivot_table()` defaults, and why matched comparisons matter. Don't show answers until I've tried.")*

> 

### 🗂️ Your record

Before you close VS Code:

1. **Run all**: The notebook should run top to bottom with no errors. The wrong table in Red Flags also runs without an error
2. Open `.specstory/history/` and confirm today's session file is there. That transcript is your evidence for the Associate audit. Your [AI Use Record](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-ai-use-record) also draws on it, as the [assessment interpretation](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Assessment/interpretation.md) explains
3. Write your `QuestLog.md` entry while it is fresh: Date, badge claimed, and one thing you would do differently

Only after that, open Debrief.ipynb for the worked answers and this week's [AI failure gallery](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-ai-failure-gallery).